# 03 - Comprehensive Multi-Dataset Exploratory Data Analysis & Compatibility

**Project**: Intelligent Driver Activity Monitoring & Safety Agent  
**Author**: Senior ML Engineer / Project Auditor  
**Datasets Analyzed**: Dataset A (Mendeley 9vr83n7z5j.2), Dataset B (Mendeley jj3tw8kj6h.3), Dataset C (Mendeley 5stn873wft.1), Dataset F (Kaggle yotamkatz)  
**Scope**: Cross-dataset quality audit, unit/coordinate inspection, class distributions, kinematic correlations, and compatibility analysis.


In [1]:
import os
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"font.size": 10, "figure.autolayout": True})

REPO_ROOT = Path(r"d:/driver-machine-learning-model")
RAW_DIR = REPO_ROOT / "dataset" / "raw"
print(f"Raw dataset root: {RAW_DIR}")


## 1. Multi-Dataset Inventory and Basic Schema Verification
Inspect physical file existence, formats, and basic schemas across primary candidate datasets.


In [2]:
inventory = {
    "Dataset A1 (Drivers)": len(list((RAW_DIR / "drivers").glob("*/*.csv"))),
    "Dataset A2 (Daywise)": len(list((RAW_DIR / "daywise").rglob("*.csv"))),
    "Dataset B (jj3tw8kj6h)": len(list((RAW_DIR / "dataset_02_jj3tw8kj6h" / "jj3tw8kj6h-3").rglob("*.csv"))),
    "Dataset C (5stn873wft)": len(list((RAW_DIR / "dataset_03_5stn873wft").glob("*.csv"))),
    "Dataset D (eishkaran)": len(list((RAW_DIR / "dataset_04_eishkaran").rglob("*.csv"))),
    "Dataset E (shashwatwork)": len(list((RAW_DIR / "dataset_05_shashwatwork").glob("*.csv"))),
    "Dataset F (yotamkatz)": len(list((RAW_DIR / "dataset_06_yotamkatz").glob("*.csv"))),
    "Dataset G (vitorrf)": len(list((RAW_DIR / "dataset_07_vitorrf").rglob("*.csv"))),
}
df_inv = pd.DataFrame(list(inventory.items()), columns=["Dataset", "CSV File Count"])
print(df_inv.to_string(index=False))


## 2. Dataset B: Multi-Class Risky Driving Maneuver Analysis
Dataset B captures 4 discrete risky maneuvers on a Raspberry Pi with MPU6050:  
- `1`: Sudden Acceleration (SA)  
- `2`: Sudden Deceleration (SD)  
- `3`: Sudden Right Turn (SRT)  
- `4`: Sudden Left Turn (SLT)  

All 1,114 raw rows represent risky events (no normal driving baseline).


In [3]:
b_raw = RAW_DIR / "dataset_02_jj3tw8kj6h" / "jj3tw8kj6h-3" / "sensor_raw.csv"
df_b = pd.read_csv(b_raw)
label_map_b = {1: "SA", 2: "SD", 3: "SRT", 4: "SLT"}
df_b["Maneuver"] = df_b["Class"].map(label_map_b)

print("Dataset B Raw Shape:", df_b.shape)
print("Class Distribution:")
print(df_b["Maneuver"].value_counts())
print("\nDriver Distribution:")
print(df_b["DriverID"].value_counts())


## 3. Dataset C: Supervised Binary Driving Style Analysis (Normal vs Aggressive)
Dataset C contains continuous 10 Hz smartphone sensor recordings with binary annotations:  
- `0`: Normal Driving  
- `1`: Aggressive Driving


In [4]:
c_final = RAW_DIR / "dataset_03_5stn873wft" / "3_FinalDatasetCsv.csv"
df_c = pd.read_csv(c_final)
df_c["Style"] = df_c["label"].map({0: "Normal Driving", 1: "Aggressive Driving"})
df_c["acc_mag"] = np.sqrt(df_c["Acc X"]**2 + df_c["Acc Y"]**2 + df_c["Acc Z"]**2)
df_c["gyro_mag"] = np.sqrt(df_c["gyro_x"]**2 + df_c["gyro_y"]**2 + df_c["gyro_z"]**2)

print("Dataset C Shape:", df_c.shape)
print("Style Distribution:")
print(df_c["Style"].value_counts())
print("\nKinematic Norm Comparison by Style:")
print(df_c.groupby("Style")[["acc_mag", "gyro_mag"]].agg(["mean", "std", "median", "max"]))


## 4. Dataset A: Data Quality, Missing Pressure & Duplicate Analysis
Demonstrate the critical finding: Pressure sensor in Drivers 3-7 contains >99% NaN values, and Driver-1 contains all 0.0 mbar.


In [5]:
press_summary = []
for d in [f"Driver-{i}" for i in range(1, 8)]:
    p = RAW_DIR / "drivers" / d / "Pressure.csv"
    if p.exists():
        pdf = pd.read_csv(p)
        press_summary.append({
            "Driver": d,
            "Total Rows": len(pdf),
            "Non-Null Rows": int(pdf["Millibars"].notnull().sum()),
            "Null Rows": int(pdf["Millibars"].isnull().sum()),
            "Null %": round(pdf["Millibars"].isnull().sum() / len(pdf) * 100, 2),
            "Values": str(pdf["Millibars"].dropna().unique()[:3])
        })
df_press = pd.DataFrame(press_summary)
print(df_press.to_string(index=False))


## 5. Cross-Dataset Compatibility Matrix & Summary
Compare sampling rates, coordinate units, gravity offsets, and suitable ML formulations.


In [6]:
compat_data = {
    "Dataset": ["Dataset A1 (Drivers)", "Dataset B (jj3tw8kj6h)", "Dataset C (5stn873wft)", "Dataset F (yotamkatz)"],
    "Hardware": ["Smartphone (Redmi/Mi)", "Raspberry Pi + MPU6050", "Smartphone (Android)", "OBD-II + GPS + Phone"],
    "Sampling Rate": ["~50 Hz (100 Hz nom)", "~1 Hz", "~10 Hz", "~1 Hz"],
    "Accel Unit": ["m/s² (includes 1g)", "g-force (~1g)", "m/s² (linear/no 1g)", "m/s²"],
    "Gyro Unit": ["rad/s", "deg/s", "rad/s", "deg/s"],
    "Labels Present": ["Driver ID (7 drivers)", "4 Risky Maneuvers", "Normal vs Aggressive", "Driver ID (4 drivers)"],
    "Designated ML Role": [
        "Driver Fingerprinting / Anomaly Detection",
        "Multi-Class Risky Maneuver Detection",
        "Binary Driving Style Classification",
        "OBD Driver Profiling"
    ]
}
df_compat = pd.DataFrame(compat_data)
print(df_compat.to_string(index=False))
